# 03 — Single Peptide Evaluation

## What this notebook does
Evaluates one candidate peptide against your frozen target spec and produces a
clear, self-contained result bundle: scores, sub-score breakdown, interpretation,
and next steps.

## What decision it helps make
> "Is this peptide worth including in a deeper scan or a stronger validation run?"

## What it cannot prove
- That the peptide binds the target
- How potent it might be
- That it is selective over other proteins
- Any biological activity

---

> **Prerequisite check:**
> This notebook is most trustworthy if notebook 02 (reference panel check) returned
> a **pass** or **degraded** status. If that notebook was never run, or if it returned
> **fail**, the scores here should not be used for any decision.
>
> Check `workspace/reference_panel/panel_interpretation.txt` before proceeding.

## Free vs Paid Colab

This notebook runs fully on **free Colab**.

| Step | Free | Paid |
|------|------|------|
| Heuristic scoring | Yes | Yes |
| Sub-score breakdown | Yes | Yes |

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "biopython", "matplotlib"])
print("Dependencies ready.")

In [ ]:
import sys, pathlib

# ── Environment detection ─────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    for _p in [
        pathlib.Path('/content/peptide-cookbooks-internal/colab-basics'),
        pathlib.Path('/content/colab-basics'),
    ]:
        if _p.exists():
            COOKBOOK_DIR = _p
            break
    else:
        raise RuntimeError(
            "Cookbook not found. Clone the repo first:\n"
            "  !git clone <your-repo-url> /content/peptide-cookbooks-internal"
        )
    WORKSPACE_DIR = pathlib.Path('/content/workspace')
else:
    COOKBOOK_DIR = pathlib.Path('..').resolve()
    WORKSPACE_DIR = COOKBOOK_DIR / 'workspace'

# ============================================================
# CONFIGURATION
# ============================================================

# Compute tier — controls size caps only. No GPU lane in this cookbook.
COMPUTE_TIER = "free"  # "free" or "paid"

# ── YOUR PEPTIDE ─────────────────────────────────────────────
YOUR_PEPTIDE_SEQUENCE = "RIEGTKLNRSFM"  # replace with your candidate
YOUR_PEPTIDE_LABEL    = "my_candidate_01"

TARGET_SPEC_PATH  = WORKSPACE_DIR / "target_spec.json"
PANEL_INTERP_PATH = WORKSPACE_DIR / "reference_panel" / "panel_interpretation.txt"
OUTPUT_DIR        = WORKSPACE_DIR / "single_eval"

print(f"Candidate: {YOUR_PEPTIDE_LABEL} — {YOUR_PEPTIDE_SEQUENCE} ({len(YOUR_PEPTIDE_SEQUENCE)} aa)")
print(f"Environment: {'Colab' if IN_COLAB else 'local'}")

In [ ]:
# Setup
import sys, pathlib, json

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(COOKBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(COOKBOOK_DIR))

from shared.target_utils import load_target_spec
from shared.scoring_utils import score_peptide

if not TARGET_SPEC_PATH.exists():
    raise FileNotFoundError(
        f"Target spec not found at {TARGET_SPEC_PATH}. Run notebook 01 first."
    )

target_spec = load_target_spec(TARGET_SPEC_PATH)
ref_sequence = target_spec["reference_policy"]["positive_control"]["sequence"]

# Panel calibration status check — same robust parser as notebook 05
if not PANEL_INTERP_PATH.exists():
    print("WARNING: panel_interpretation.txt not found. Run notebook 02 first.")
    print("Scores below are not trustworthy without a passing calibration check.")
else:
    _interp_text = PANEL_INTERP_PATH.read_text()
    _status_line = next(
        (l.strip() for l in _interp_text.splitlines()
         if l.strip().lower().startswith("panel status:")),
        None,
    )
    if _status_line is None:
        print("WARNING: Could not parse panel status. Re-run notebook 02.")
    else:
        _panel_status = _status_line.split(":", 1)[1].strip().lower()
        if _panel_status == "fail":
            print("WARNING: Reference panel check returned FAIL.")
            print("Scores from this notebook are not trustworthy for ranking.")
            print("Fix pocket hints in notebook 01 and re-run notebook 02 before continuing.")
        else:
            print(f"Panel calibration status: {_panel_status}")

print(f"\nTarget: {target_spec['pdb_id']} chain {target_spec['chain_id']}")
print(f"Reference sequence: {ref_sequence}")

## Step 1 — Score the Candidate

In [ ]:
result = score_peptide(YOUR_PEPTIDE_SEQUENCE, target_spec)

if result.get('error'):
    print(f"ERROR: {result['interpretation']}")
else:
    print(f"Candidate: {YOUR_PEPTIDE_LABEL}")
    print(f"Sequence:  {result['sequence']}")
    print(f"Length:    {result['sub_scores']['length']}")
    print()
    print(f"Composite Score: {result['composite_score']:.4f}")
    print(f"Lane: {result['lane']}")
    print()
    print("Sub-scores:")
    for key, val in result['sub_scores'].items():
        print(f"  {key:<25}: {val}")
    print()
    print("Interpretation:")
    print(f"  {result['interpretation']}")

## Step 2 — Compare Against Reference and Controls

In [ ]:
from shared.scoring_utils import score_panel
from shared.panel_utils import make_reference_panel

comparison_panel = make_reference_panel(
    positive_control=ref_sequence,
    positive_control_label="reference_positive",
    poly_ala_length=len(YOUR_PEPTIDE_SEQUENCE),
)
# Insert the candidate
comparison_panel.insert(0, {
    "label": YOUR_PEPTIDE_LABEL,
    "sequence": YOUR_PEPTIDE_SEQUENCE,
    "role": "candidate",
    "description": "Candidate peptide under evaluation.",
})

comparison_results = score_panel(comparison_panel, target_spec)

print("Comparison (ranked):")
print(f"{'Label':<25} {'Score':>7} {'Role':<22} {'Sequence'}")
print("-" * 80)
for r in comparison_results:
    marker = " <-- YOUR CANDIDATE" if r['label'] == YOUR_PEPTIDE_LABEL else ""
    print(f"{r['label']:<25} {float(r.get('composite_score', 0)):>7.4f} "
          f"{r.get('role', ''):<22} {r['sequence']}{marker}")

## Step 3 — Sub-Score Breakdown Chart

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sub = result['sub_scores']

# The composite has three sub-scores: length, charge, hydrophobic (weights 0.30/0.40/0.30).
# BLOSUM similarity to the reference is a diagnostic only — it is NOT in the composite.
sub_keys   = ['length_score', 'charge_score', 'hydrophobic_score']
sub_labels = ['Length fit (w=0.30)', 'Charge match (w=0.40)', 'Hydrophobic match (w=0.30)']
weights    = [0.30, 0.40, 0.30]
sub_vals   = [float(sub.get(k, 0)) for k in sub_keys]
weighted_vals = [v * w for v, w in zip(sub_vals, weights)]

blosum_diag = float(sub.get('reference_similarity_blosum', 0))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
colors_sub = ['#3498db', '#e67e22', '#27ae60']

# Raw sub-scores
ax1 = axes[0]
ax1.barh(sub_labels, sub_vals, color=colors_sub, edgecolor='white')
ax1.set_xlim(0, 1)
ax1.set_title(f"Sub-Scores — {YOUR_PEPTIDE_LABEL}", fontsize=10)
ax1.set_xlabel("Score (0–1)")
ax1.axvline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.7)
for i, v in enumerate(sub_vals):
    ax1.text(v + 0.01, i, f"{v:.3f}", va='center', fontsize=8)

# Weighted contributions — sum should equal composite_score
ax2 = axes[1]
ax2.barh(sub_labels, weighted_vals, color=colors_sub, edgecolor='white')
ax2.set_title(f"Weighted (sum = {sum(weighted_vals):.4f} = composite)", fontsize=10)
ax2.set_xlabel("Weighted score")
for i, v in enumerate(weighted_vals):
    ax2.text(v + 0.002, i, f"{v:.3f}", va='center', fontsize=8)

# BLOSUM diagnostic (separate panel, clearly labelled)
ax3 = axes[2]
ax3.bar(['BLOSUM\nsimilarity\nto reference'], [blosum_diag],
        color='#95a5a6', edgecolor='white', width=0.4)
ax3.set_ylim(0, 1)
ax3.set_title("Reference similarity\n(diagnostic — not in composite)", fontsize=9)
ax3.set_ylabel("Similarity (0–1)")
ax3.text(0, blosum_diag + 0.02, f"{blosum_diag:.3f}", ha='center', fontsize=9)
ax3.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)

plt.suptitle(f"Heuristic Lane — {target_spec['pdb_id']} chain {target_spec['chain_id']}",
             fontsize=10, y=1.02)
plt.tight_layout()
chart_path = OUTPUT_DIR / "sub_score_breakdown.png"
plt.savefig(chart_path, dpi=120, bbox_inches='tight')
plt.show()
print(f"Saved to {chart_path}")
print(f"\nComposite score check: {sum(weighted_vals):.4f} (should match {result['composite_score']:.4f})")

## Step 4 — Save Result Bundle

In [ ]:
import json as _json

score = result['composite_score']
ref_result = next((r for r in comparison_results if r['label'] == 'reference_positive'), None)
ref_score = float(ref_result.get('composite_score', 0.0)) if ref_result else None

# Suggested action is based on the absolute pocket-fitness score only.
# The reference comparison is a diagnostic note — it shows how similar the candidate's
# pocket-property character is to the reference, not how similar their binding might be.
# Do not use the reference ratio as a decision trigger: the reference score on the
# heuristic lane reflects pocket-property matching, not biological activity.
if score >= 0.70:
    suggested_action = "consider_further_evaluation"
    action_reason = (
        "Passes pocket-property checks at a reasonable level. "
        "Worth scanning variants (notebook 04) or proceeding to notebook 05."
    )
elif score >= 0.50:
    suggested_action = "consider_with_caution"
    action_reason = (
        "Moderate pocket-property match. May benefit from variant scanning. "
        "If this seems unexpectedly low, revisit pocket hints in notebook 01."
    )
else:
    suggested_action = "review_design"
    action_reason = (
        "Low pocket-property match. Reconsider the sequence design, or review "
        "POCKET_NET_CHARGE_HINT and POCKET_HYDROPHOBIC_HINT in notebook 01."
    )

# Reference comparison: context only, not an action trigger.
if ref_score and ref_score > 0:
    ref_comparison_note = (
        f"candidate={score:.4f}, reference={ref_score:.4f}, "
        f"ratio={score/ref_score:.2f}x — "
        "ratio reflects pocket-property similarity to reference sequence, not binding comparison"
    )
else:
    ref_comparison_note = "reference score unavailable"

bundle = {
    "candidate_label": YOUR_PEPTIDE_LABEL,
    "candidate_sequence": YOUR_PEPTIDE_SEQUENCE,
    "target_pdb_id": target_spec["pdb_id"],
    "target_chain_id": target_spec["chain_id"],
    "composite_score": result['composite_score'],
    "sub_scores": result['sub_scores'],
    "interpretation": result['interpretation'],
    "reference_comparison_diagnostic": ref_comparison_note,
    "lane": result['lane'],
    "suggested_action": suggested_action,
    "action_reason": action_reason,
    "caveats": [
        "Heuristic lane only — not a physics score or docking result.",
        "Score does not imply binding affinity.",
        "Reference panel calibration status should be 'pass' before trusting this ranking.",
        "Suggested action is based on absolute pocket-fitness score, not reference comparison.",
    ],
}

bundle_path = OUTPUT_DIR / "eval_result.json"
bundle_path.write_text(_json.dumps(bundle, indent=2))
print(f"Result bundle saved to {bundle_path}")
print()
print(f"Suggested action : {suggested_action}")
print(f"Reason           : {action_reason}")
print(f"Reference context: {ref_comparison_note}")

## Interpreting the Result

### What the score means at this stage

| Score range | Interpretation |
|-------------|----------------|
| ≥ 0.70 | Passes basic property checks. Worth including in a scan. |
| 0.50–0.70 | Moderate match. Scanning variants may improve it. |
| 0.30–0.50 | Weak match. Consider a different design approach. |
| < 0.30 | Poor match. Likely wrong composition for this pocket. |

### What the score does NOT mean

- It is not a binding constant (Kd, Ki, IC50)
- It is not a docking score or a free energy estimate
- A high score does not mean the peptide will be active in a cell assay
- This is **Claims Level 1** (cheap heuristic) from the repo's Claims Ladder

To make a stronger claim, this peptide would need to:
1. Pass notebook 04 (SAR scan — shows it is at or near a local optimum)
2. Pass notebook 05 (promoted to strong validation)
3. Survive a structure-prediction or physics-based lane (e.g., AF2, Boltz, PyRosetta)

## Outputs

| File | Description |
|------|-------------|
| `workspace/single_eval/eval_result.json` | Full result bundle with scores and interpretation |
| `workspace/single_eval/sub_score_breakdown.png` | Breakdown chart |

## Next Notebook

→ **04_small_sar_scan.ipynb** — scan variants of this sequence to find local improvements

OR

→ **05_promote_to_strong_validation.ipynb** — if you are satisfied and ready to shortlist